# §6.1 default unit — summer and winter design (M1-10)

Solves AHU-1 from `docs/SPEC.md` §6.1 (with the components drafted in
`docs/decisions/0002`) at a summer and a winter design condition and prints the
state at every point in the unit, in I-P.

**Read this first**

- **Design conditions are illustrative**, not ASHRAE design values (their
  licensing is open question §14-3): summer OA 91 °F db / 74 °F wb with RA
  75 °F / 50 %; winter OA 13 °F / 50 % with RA 70 °F / 30 %.
- **Actuator positions are set by hand.** The control loops arrive in M2; here a
  simple search sets the OA damper for 3,000 cfm *at the OA damper* (the
  location the config states) and the coils are opened or closed explicitly.
- Engine is SI inside; everything below is converted for display. I-P enthalpy
  is computed with I-P equations, not h_SI / 2.326.

In [1]:
from pathlib import Path

from ahuverify import units
from ahuverify._numeric import bisect
from ahuverify.lanes import compile_unit
from ahuverify.schema import UnitConfig
from ahuverify.solver import solve
from ahuverify.state import AirState

CONFIG = Path("..") / "tests" / "fixtures" / "example_6_1.json"
unit = compile_unit(UnitConfig.model_validate_json(CONFIG.read_text()))
P = unit.p
print(f"Site pressure {P:.0f} Pa (standard atmosphere at 33 ft)")

Site pressure 101204 Pa (standard atmosphere at 33 ft)


In [2]:
def F(t_f):
    return units.to_si(t_f, "F")


def cfm(stream):
    return units.from_si(stream.m_da * stream.state.v, "cfm")


def oa_damper_cfm(result):
    """OA flow at the OA damper (after the wheel), where min OA is stated."""
    m_oa = result.components["mix1"].loads["m_oa"]
    return units.from_si(m_oa * result.states["after:erw1.supply"].state.v, "cfm")


def set_min_oa(oa, ra, actuators, target_cfm=3000.0):
    """Search the OA damper position that gives target_cfm at the OA damper."""

    def gap(f):
        return (
            oa_damper_cfm(solve(unit, oa, ra, {**actuators, "mix1.oa_fraction": f}))
            - target_cfm
        )

    return bisect(gap, 0.0, 1.0, tol=1e-6)


def state_table(result):
    print(
        f"{'Location':22} {'db °F':>7} {'wb °F':>7} {'RH %':>6} {'W gr/lb':>8} {'h Btu/lb':>9} {'cfm':>8}"
    )
    for key, s in result.states.items():
        st = s.state
        print(
            f"{key:22} {units.from_si(st.t_db, 'F'):7.1f} {units.from_si(st.t_wb, 'F'):7.1f} "
            f"{st.rh * 100:6.0f} {st.w * 7000:8.1f} {units.enthalpy_ip(st.t_db, st.w):9.2f} {cfm(s):8.0f}"
        )


def failed_checks(result):
    bad = [
        (n, c) for n, r in result.components.items() for c in r.checks if not c.passed
    ]
    for name, c in bad:
        if c.unit == "m/s":
            print(
                f"FAIL {name}.{c.name}: {units.from_si(c.value, 'fpm'):.0f} fpm vs {units.from_si(c.limit, 'fpm'):.0f} fpm"
            )
        elif c.unit == "m3/s":
            print(
                f"FAIL {name}.{c.name}: {units.from_si(c.value, 'cfm'):.0f} cfm vs {units.from_si(c.limit, 'cfm'):.0f} cfm"
            )
        else:
            print(f"FAIL {name}.{c.name}: {c.value:.4g} vs {c.limit:.4g} {c.unit}")
    if not bad:
        print("All component checks pass.")

## Summer design

Cooling coil wide open, heating coils off, minimum OA. Target SAT would be
55 °F — see whether the unit gets there.

In [3]:
oa = AirState.from_db_wb(F(91.0), F(74.0), P)
ra = AirState.from_db_rh(F(75.0), 0.50, P)
acts = {"cc1.valve": 1.0, "phc1.valve": 0.0, "rhc1.valve": 0.0}
acts["mix1.oa_fraction"] = set_min_oa(oa, ra, acts)
summer = solve(unit, oa, ra, acts)
print(f"Converged in {summer.passes} passes; valid: {summer.valid}")
print(
    f"OA damper position {acts['mix1.oa_fraction'] * 100:.1f} % -> {oa_damper_cfm(summer):.0f} cfm at the damper\n"
)
state_table(summer)
cc = summer.components["cc1"].loads
print(
    f"\ncc1: {cc['q_total'] / 3516.85:.1f} tons, SHR {cc['shr']:.2f}, "
    f"condensate {cc['condensate'] * 7936.64:.0f} lb/h, implied CHWR {units.from_si(cc['chwr_implied'], 'F'):.1f} °F"
)
sat = units.from_si(summer.states["after:sf1"].state.t_db, "F")
print(
    f"SAT {sat:.1f} °F with the valve wide open: {sat - 55.0:.1f} °F short of 55 °F\n"
)
failed_checks(summer)

Converged in 6 passes; valid: True
OA damper position 28.8 % -> 3000 cfm at the damper

Location                 db °F   wb °F   RH %  W gr/lb  h Btu/lb      cfm
ra                        75.0    62.6     50     64.7     28.12     9800
after:rf1                 76.0    62.9     48     64.7     28.37     9819
after:mix1.relief         76.0    62.9     48     64.7     28.37     2412
after:erw1.exhaust        87.2    70.8     45     86.9     34.58     2474
after:ef1                 87.9    71.0     44     86.9     34.75     2478
oa_intake                 91.0    74.0     45     99.1     37.43     3063
after:erw1.supply         81.9    68.1     50     81.0     32.36     3000
after:mix1                77.7    64.5     49     69.4     29.51    10407
after:flt1                77.7    64.5     49     69.4     29.51    10407
after:phc1                77.7    64.5     49     69.4     29.51    10407
after:cc1                 54.8    53.7     94     59.8     22.43     9942
after:rhc1              

## Winter design

All coils off, minimum OA; the energy wheel does the heating.

In [4]:
oa = AirState.from_db_rh(F(13.0), 0.50, P)
ra = AirState.from_db_rh(F(70.0), 0.30, P)
acts = {"cc1.valve": 0.0, "phc1.valve": 0.0, "rhc1.valve": 0.0}
acts["mix1.oa_fraction"] = set_min_oa(oa, ra, acts)
winter = solve(unit, oa, ra, acts)
print(f"Converged in {winter.passes} passes; valid: {winter.valid}")
print(
    f"OA damper position {acts['mix1.oa_fraction'] * 100:.1f} % -> {oa_damper_cfm(winter):.0f} cfm at the damper\n"
)
state_table(winter)
print()
failed_checks(winter)

Converged in 8 passes; valid: True
OA damper position 30.2 % -> 3000 cfm at the damper

Location                 db °F   wb °F   RH %  W gr/lb  h Btu/lb      cfm
ra                        70.0    53.0     30     32.6     21.88     9800
after:rf1                 71.0    53.4     29     32.6     22.13     9819
after:mix1.relief         71.0    53.4     29     32.6     22.13     2877
after:erw1.exhaust        28.2    25.2     68     15.0      9.08     2635
after:ef1                 28.9    25.6     66     15.0      9.23     2638
oa_intake                 13.0    10.2     50      5.3      3.93     2757
after:erw1.supply         52.7    41.7     37     21.5     15.98     3000
after:mix1                65.3    50.0     32     29.2     20.22     9942
after:flt1                65.3    50.0     32     29.2     20.22     9942
after:phc1                65.3    50.0     32     29.2     20.22     9942
after:cc1                 65.3    50.0     32     29.2     20.22     9942
after:rhc1              

## What this shows

- **Summer: the unit cannot make 55 °F supply air.** cc1 is rated for 55 °F
  leaving, and the draw-through supply fan (4.0 in. w.c. plus 1.0 in. w.c. dirty
  filter, motor in the airstream) adds about 3 °F after it. M2 will report this
  as `SETPOINT_NOT_MET` on the cooling loop.
- **The 4.5 ft² return damper is over 1,500 fpm even at minimum OA**, not only
  in warmup (F-12).
- **Winter: the wheel alone lifts mixed air well above 55 °F**, so the sequence
  will need economizer or wheel modulation, not heat, at this condition.
- The wheel's exhaust leaves at about 28 °F — below freezing, but only ~68 % RH,
  so not saturated: no frost by the spec §5.4 criterion at 13 °F OA.